In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping


df = pd.read_csv("/content/sleep_apnea_synthetic_data.csv")

# Print DataFrame columns to help diagnose missing 'severity_class'
print("DataFrame columns:", df.columns.tolist())

FEATURE_COLS = [
    "accel_mean", "accel_std", "motion_count", "zero_crossing_rate",
    "hr_mean", "hr_std", "hr_delta", "hr_trend",
    "spo2_mean", "spo2_min", "desaturation_depth", "desaturation_flag", "recovery_time",
    "motion_spo2_lag", "hr_spo2_correlation"
]
LABEL_COL = "severity_label" # Changed from 'severity_class' to 'severity_label'
GROUP_COL = "subject_id"

# Filter FEATURE_COLS to include only columns present in the DataFrame
existing_features = [col for col in FEATURE_COLS if col in df.columns]

scaler = StandardScaler()
df[existing_features] = scaler.fit_transform(df[existing_features])

# Update FEATURE_COLS to reflect only the existing features
FEATURE_COLS = existing_features


label_encoder = LabelEncoder()
subject_labels = df.groupby(GROUP_COL)[LABEL_COL].first()
subject_labels_encoded = label_encoder.fit_transform(subject_labels)
num_classes = len(label_encoder.classes_)


sequences = []
labels = []

for subject_id, group in df.groupby(GROUP_COL):
    group = group.sort_values("epoch_index")
    seq = group[FEATURE_COLS].values # Use the updated FEATURE_COLS
    sequences.append(seq)
    labels.append(label_encoder.transform([group[LABEL_COL].iloc[0]])[0])

MAX_SEQ_LEN = 200
X = pad_sequences(sequences, maxlen=MAX_SEQ_LEN, dtype="float32", padding="post", truncating="post")
y = to_categorical(labels, num_classes=num_classes)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.176, random_state=42,  # ~15% of total
    stratify=np.argmax(y_train, axis=1)
)


model = Sequential([
    Masking(mask_value=0.0, input_shape=(MAX_SEQ_LEN, len(FEATURE_COLS))), # Use the updated FEATURE_COLS
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)


test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))
print(confusion_matrix(y_true, y_pred))


model.save("apnea_severity_lstm_model.h5")

DataFrame columns: ['subject_id', 'night_id', 'epoch_index', 'age', 'bmi', 'acc_mean', 'acc_std', 'hr_mean', 'hr_std', 'spo2_mean', 'spo2_min', 'is_apnea_event', 'severity_label']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 200, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 200, 64)        │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 200, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,268 (122.14 KB)

 Trainable params: 31,268 (122.14 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 411ms/step - accuracy: 0.3856 - loss: 1.3501 - val_accuracy: 0.4848 - val_loss: 1.2947
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 228ms/step - accuracy: 0.4444 - loss: 1.2931 - val_accuracy: 0.4848 - val_loss: 1.2049
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 221ms/step - accuracy: 0.4444 - loss: 1.2169 - val_accuracy: 0.4848 - val_loss: 1.1140
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 212ms/step - accuracy: 0.4510 - loss: 1.1492 - val_accuracy: 0.4848 - val_loss: 1.0452
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 214ms/step - accuracy: 0.4641 - loss: 1.1043 - val_accuracy: 0.4848 - val_loss: 1.0084
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 361ms/step - accuracy: 0.4837 - loss: 1.0842 - val_accuracy: 0.5152 - val_loss: 0.9808
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 229ms/step - accuracy: 0.4837 - loss: 1.0486 - val_accuracy: 0.5152 - val_loss: 0.9586
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 207ms/step - accuracy: 0.5229 - loss: 1.0133 - val_accuracy: 0.5152 - v

              precision    recall  f1-score   support

        mild       0.67      0.60      0.63        10
    moderate       0.33      0.29      0.31         7
        none       0.75      1.00      0.86         9
      severe       0.67      0.57      0.62         7

    accuracy                           0.64        33
   macro avg       0.60      0.61      0.60        33
weighted avg       0.62      0.64      0.62        33

[[6 1 3 0]
 [3 2 0 2]
 [0 0 9 0]
 [0 3 0 4]]
